# GIADA IV-C2 — plasticità e rilascio stocastico
Contratto preregistrato, teacher canonico e seed appaiati. Il risultato non autorizza IV-C3 o Task33. Output compatto per evitare flooding dell'interfaccia Kaggle.


In [ ]:
from pathlib import Path
import json,subprocess,sys,tempfile,uuid
from IPython.display import display
WORK=Path(tempfile.mkdtemp(prefix='giada_iv_c2_'))
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','--quiet','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','--quiet','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--quiet','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7'
print({'revision':REVISION,'neuron':neuron.__version__})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_iv_c2_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
command=[sys.executable,'-u',str(REPO/'scripts/run_iv_c2_stochastic_release.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
completed=subprocess.run(command,cwd=REPO,check=False)
result=next((OUTPUT/p for p in ('final_report.json','failure_report.json') if (OUTPUT/p).is_file()),None)
report=json.loads(result.read_text()) if result else {'valid':False,'error':f'worker interrupted (exit={completed.returncode})'}
display({k:report.get(k) for k in ('valid','case_count','case_gate_passed','distribution_gate_passed','uniform_negative_control_passed','iv_c2_passed','iv_c3_passed','task33_authorized','error')})
if report.get('cases'):
 display({'max_state_error':max(x['max_state_error'] for x in report['cases']),'max_rng_sequence_difference':max(x['max_rng_sequence_difference'] for x in report['cases']),'release_mismatches':sum(x['release_mismatch_count'] for x in report['cases'])})
print({'output':str(OUTPUT),'exit':completed.returncode})
